# Mosaicing Example

This notebook demonstrates how to use PyFibreBundle for mosaicing using the `Mosaic` class.

## Overview
1. Load and calibrate a fibre bundle video
2. Pre-process each frame (crop, filter, mask)
3. Build the mosaic iteratively using `Mosaic.add()`
4. Display the final mosaic

## Setup: Import Libraries

In [ ]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2 as cv

import context

import pybundle
from pybundle import Mosaic

## Load Video and Calibration Image

In [ ]:
filterSize = 1.5      # Size of Gaussian filter applied to each frame before mosaicing

cap = cv.VideoCapture(str(Path('../test/data/raw_example.avi')))
ret, img = cap.read()
img = img[:, :, 0]
nFrames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))

print(f'Video loaded: {nFrames} frames, frame shape: {img.shape}')

In [ ]:
calibImg = np.array(Image.open(Path('../test/data/raw_example_calib.tif')))
print(f'Calibration image shape: {calibImg.shape}')

## Bundle Calibration

Use the calibration image to find the bundle position and build a mask.

In [ ]:
loc = pybundle.find_bundle(calibImg)
mask = pybundle.get_mask(calibImg, loc)

print(f'Bundle location: centre ({loc[0]:.0f}, {loc[1]:.0f}), radius {loc[2]:.0f} px')

## Pre-process All Frames

`crop_filter_mask` crops the image to the bundle, applies a Gaussian filter, and applies the circular mask.

In [ ]:
# Pre-process first frame to get output shape
cap.set(cv.CAP_PROP_POS_FRAMES, 0)
ret, img = cap.read()
img = img[:, :, 0]
img = pybundle.crop_filter_mask(img, loc, mask, filterSize)

imgStack = np.zeros([nFrames, img.shape[0], img.shape[1]], dtype='uint8')
imgStack[0, :, :] = img

# Pre-process remaining frames
for i in range(1, nFrames):
    cap.set(cv.CAP_PROP_POS_FRAMES, i)
    ret, img = cap.read()
    img = img[:, :, 0]
    img = pybundle.crop_filter_mask(img, loc, mask, filterSize)
    imgStack[i, :, :] = img

print(f'Pre-processed stack shape: {imgStack.shape}')

In [ ]:
# Show the first processed frame
plt.figure(dpi=120)
plt.imshow(imgStack[0], cmap='gray')
plt.title('First Processed Frame')
plt.axis('off')
plt.show()

## Build the Mosaic

Create a `Mosaic` object and add each pre-processed frame. The mosaic canvas is 1000×1000 px; each incoming frame is resized to 250 px before insertion.

In [ ]:
mosaic = Mosaic(1000, resize=250)

t0 = time.time()

for i in range(nFrames):
    mosaic.add(imgStack[i])

elapsed = time.time() - t0
print(f'Average time per frame: {1000 * elapsed / nFrames:.1f} ms')

## Display the Final Mosaic

In [ ]:
m = mosaic.get_mosaic()

plt.figure(figsize=(7, 7), dpi=120)
plt.imshow(m, cmap='gray')
plt.title('Final Mosaic')
plt.axis('off')
plt.show()

print(f'Mosaic shape: {m.shape}')

## Incremental Mosaic Visualisation

Replay the mosaicing process and display every 10th frame update to show how the mosaic grows.

In [ ]:
mosaic2 = Mosaic(1000, resize=250)

snapshots = []
snap_every = max(1, nFrames // 10)

for i in range(nFrames):
    mosaic2.add(imgStack[i])
    if i % snap_every == 0 or i == nFrames - 1:
        snapshots.append((i + 1, mosaic2.get_mosaic().copy()))

cols = min(len(snapshots), 3)
rows = (len(snapshots) + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows), dpi=80)
axes = np.array(axes).flatten()

for ax, (frame_idx, snap) in zip(axes, snapshots):
    ax.imshow(snap, cmap='gray')
    ax.set_title(f'After frame {frame_idx}')
    ax.axis('off')

for ax in axes[len(snapshots):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()